In [ ]:
!pip install -qU "langchain[openai]" python-dotenv

In [1]:
import os
from dotenv import load_dotenv

# .env 파일에서 API 키를 자동으로 불러옵니다
load_dotenv()

True

## Runtime & State (런타임과 상태)

**Runtime**은 에이전트 실행 중에 사용할 수 있는 실행 환경 정보입니다.  
**Context**를 사용하면 사용자 정보, 설정값 등을 에이전트 실행 흐름 전체에 주입할 수 있습니다.

**미들웨어 실행 시점 비교:**

| 데코레이터 | 실행 시점 | 주요 용도 |
|------------|----------|----------|
| `@before_model` | LLM 호출 **직전** | 로깅, 상태 검사, 프롬프트 수정 |
| `@wrap_model_call` | LLM 호출 **전후 포함** | 모델 교체, 요청 변환 |
| `@after_model` | LLM 호출 **직후** | 응답 검사, 후처리 |
| `@before_agent` | 에이전트 실행 전 | 입력 검증, 가드레일 |
| `@after_agent` | 에이전트 실행 후 | 출력 검증, 안전 필터 |

### node-style

In [2]:
# dataclass로 에이전트에 전달할 Context(문맥 정보)를 정의합니다
# Context는 미들웨어와 에이전트 전체에서 runtime.context로 접근 가능합니다
from dataclasses import dataclass

@dataclass
class Context:
    user_name: str  # 현재 사용자 이름 — 미들웨어에서 runtime.context.user_name으로 접근


In [3]:
from langchain.agents.middleware import before_model

# @before_model: 이 함수는 LLM이 호출되기 직전에 자동으로 실행됩니다
# state: 현재 대화 상태 (messages 등)
# runtime: 실행 환경 (context, store, config 등)
@before_model
def log_before_model(state, runtime):
    print(f'(step1)before_model_state : \n{state}')   # 현재 대화 상태 출력
    print(f'(step2)before_model_runtime : \n{runtime}')  # 런타임 환경 출력
    print(f'(step3)사용자 이름 : {runtime.context.user_name}')  # Context에서 사용자 이름 추출
    return None  # None 반환 시 에이전트 흐름을 그대로 유지합니다


In [4]:
from langchain.agents import create_agent

# context_schema=Context 를 지정하면
# agent.invoke() 시 context 인자를 통해 Context 객체를 전달할 수 있습니다
agent = create_agent(
    model="gpt-5-mini",
    tools=[],
    middleware=[log_before_model],  # before_model 미들웨어 등록
    context_schema=Context          # Context 스키마 지정
)



In [5]:
# context=Context(user_name='Jay') 로 사용자 정보를 에이전트에 전달합니다
# 미들웨어에서 runtime.context.user_name 으로 'Jay'에 접근 가능합니다
agent.invoke(
    {"messages": [{"role": "user", "content": "제 이름이 뭐죠?"}]},
    context=Context(user_name='Jay')
)

(step1)before_model_state : 
{'messages': [HumanMessage(content='제 이름이 뭐죠?', additional_kwargs={}, response_metadata={}, id='cd6771f4-da2a-46cd-8630-4f82d361a887')]}
(step2)before_model_runtime : 
Runtime(context=Context(user_name='Jay'), store=None, stream_writer=<function Pregel.stream.<locals>.stream_writer at 0x00000234372DDDA0>, previous=None, execution_info=ExecutionInfo(node_attempt=1, node_first_attempt_time=1777959337.2362094))
(step3)사용자 이름 : Jay


{'messages': [HumanMessage(content='제 이름이 뭐죠?', additional_kwargs={}, response_metadata={}, id='cd6771f4-da2a-46cd-8630-4f82d361a887'),
  AIMessage(content='모르겠어요 — 당신이 이름을 알려주지 않으셨기 때문입니다. 이름을 알려주시면 그 이름으로 불러드릴게요. 원하시면 닉네임이나 호칭(예: ~님, ~씨)도 알려주세요. (참고: 이 대화 중에는 기억할 수 있지만 다음 대화엔 다시 알려주셔야 할 수 있어요.)', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 541, 'prompt_tokens': 12, 'total_tokens': 553, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 448, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-Dc2jTUe2UfY3ix5FEl0CaiA3AAGiK', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019df6a2-7f88-7c10-a8c8-3986f99dde09-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 12, 'output_tok

### wrap-style

`@wrap_model_call`은 LLM 호출 전후를 **감싸는(wrapping)** 방식입니다.  
`handler(request)`를 직접 호출하여 LLM을 실행하므로, 요청(request)을 수정하고 응답을 가로챌 수 있습니다.

```python
@wrap_model_call
def my_middleware(request, handler):
    # 1. request 수정 (모델 교체, 시스템 프롬프트 추가 등)
    new_request = request.override(system_prompt='...')
    # 2. handler로 LLM 호출
    return handler(new_request)
```

In [6]:
from langchain.agents.middleware import wrap_model_call
from langchain.messages import HumanMessage, SystemMessage

# @wrap_model_call: LLM 호출을 감싸는 미들웨어
# request: LLM에 전달될 요청 객체 (messages, model, config 포함)
# handler: 실제 LLM을 호출하는 함수 — 반드시 호출해야 합니다
@wrap_model_call
def inject_user_name(request, handler):
    print(f'Request : {request}')
    user_name = request.runtime.context.user_name  # Context에서 사용자 이름 추출
    if user_name:
        # request.override()로 시스템 프롬프트를 동적으로 삽입합니다
        system_message = f'사용자의 이름은 {user_name}입니다'
        request = request.override(system_prompt=system_message)
    return handler(request)  # 수정된 request로 LLM을 호출합니다


In [7]:
from langchain.agents.middleware import after_model

# @after_model: LLM 응답 직후에 실행되는 미들웨어
# state['messages'][-1]: 방금 생성된 AI 응답 메시지
@after_model
def log_after_model(state, runtime):
    print(f"모델 응답 완료 후 : {state['messages'][-1].content}")
    return None  # None 반환 시 에이전트 흐름을 그대로 유지합니다

In [8]:
# inject_user_name과 log_after_model 두 미들웨어를 함께 등록합니다
# 미들웨어는 리스트 순서대로 실행됩니다
agent = create_agent(
    model="gpt-5-mini",
    tools=[],
    middleware=[inject_user_name, log_after_model],
    context_schema=Context
)

In [9]:
# 'Jay Pak'이라는 이름이 시스템 프롬프트에 주입되어 에이전트가 이름을 알게 됩니다
agent.invoke(
    {"messages": [{"role": "user", "content": "제 이름이 뭐죠?"}]},
    context=Context(user_name='Jay Pak')
)

Request : ModelRequest(model=ChatOpenAI(profile={'max_input_tokens': 272000, 'max_output_tokens': 128000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x0000023437279210>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x00000234372D7090>, root_client=<openai.OpenAI object at 0x000002344D58A810>, root_async_client=<openai.AsyncOpenAI object at 0x0000023437278C90>, model_name='gpt-5-mini', model_kwargs={}, openai_api_key=SecretStr('**********'), stream_usage=True), messages=[HumanMessage(content='제 이름이 뭐죠?', additional_kwargs={}, response_metad

{'messages': [HumanMessage(content='제 이름이 뭐죠?', additional_kwargs={}, response_metadata={}, id='08e8aacb-7b71-42cd-bf1f-1fd8db97c7f5'),
  AIMessage(content='Jay Pak님입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 206, 'prompt_tokens': 23, 'total_tokens': 229, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 192, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-Dc2jnLCO8q9c14qjdMtixv55Yfsll', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019df6a2-cfdd-77a3-9d57-7d8204784231-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 23, 'output_tokens': 206, 'total_tokens': 229, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reas

### 실습 — 멤버십 등급에 따른 동적 모델 선택

Context에 `is_premium` 플래그를 추가하여 사용자 등급에 따라 다른 LLM 모델을 사용합니다.  
프리미엄 사용자 → GPT-5 (고성능), 일반 사용자 → GPT-5-nano (경제적)

In [10]:
# Context에 멤버십 정보를 추가합니다
@dataclass
class Context:
    user_name: str
    is_premium: bool = False  # True=프리미엄 회원, False=일반 회원 (기본값)


In [11]:
from langchain.chat_models import init_chat_model

@wrap_model_call
def dynamic_model_selector(request, handler):
    user_name = request.runtime.context.user_name
    is_premium = request.runtime.context.is_premium  # 멤버십 등급 확인

    if is_premium:
        model_name = 'gpt-5'  # 프리미엄: 고성능 모델
        print(f'{user_name} 사용자는 프리미엄 회원입니다. GPT-5를 호출합니다.')
    else:
        model_name = 'gpt-5-nano'  # 일반: 경제적 모델
        print(f'{user_name} 사용자는 일반 회원입니다. GPT-5-nano를 호출합니다.')

    # 새 모델 객체를 생성하고 request에 적용합니다
    new_model = init_chat_model(model_name)
    new_request = request.override(model=new_model)  # 모델 교체

    return handler(new_request)


In [12]:
agent = create_agent(
    model='gpt-5-nano',        # 기본 모델 (미들웨어에서 동적으로 교체됩니다)
    tools=[],
    middleware=[dynamic_model_selector],
    context_schema=Context
)

In [13]:
# 프리미엄 회원 — GPT-5로 자동 전환됩니다
agent.invoke(
    {"messages": [{"role": "user", "content": "안녕하세요"}]},
    context=Context(user_name='Jay', is_premium=True)
)

Jay 사용자는 프리미엄 회원입니다. GPT-5를 호출합니다.


{'messages': [HumanMessage(content='안녕하세요', additional_kwargs={}, response_metadata={}, id='69e9e32c-1fb2-4bfd-afff-03e98c5b9b59'),
  AIMessage(content='안녕하세요! 무엇을 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 84, 'prompt_tokens': 8, 'total_tokens': 92, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 64, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DM6BiITwzhnHP7Qqchf4hCXChVI5M', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d1423-a502-7d20-8c4d-f116afd1827c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 84, 'total_tokens': 92, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 64

In [14]:
# 일반 회원 — GPT-5-nano 유지됩니다
agent.invoke(
    {"messages": [{"role": "user", "content": "안녕하세요"}]},
    context=Context(user_name='Jay', is_premium=False)
)

Jay 사용자는 일반 회원입니다. GPT-5-nano를 호출합니다.


{'messages': [HumanMessage(content='안녕하세요', additional_kwargs={}, response_metadata={}, id='0c48be41-37b2-480d-8d1f-e9bd3e38df66'),
  AIMessage(content='안녕하세요! 반갑습니다. 무엇을 도와드릴까요? 궁금한 점이나 필요하신 도움을 말씀해 주세요. 예) 번역, 글쓰기/요약, 정보 검색, 코딩 도움, 일정 정리 등. 어떤 주제로 시작할까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 324, 'prompt_tokens': 8, 'total_tokens': 332, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 256, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DM6Bn7I5c6WoKAb6DZUZEQVTAkFwg', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d1423-b71c-7d13-b5dd-e50b5c7a7aa6-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 324, 'total_tokens': 332, 'input_toke

## Custom Middleware (커스텀 미들웨어)

Built-in 미들웨어 외에도 원하는 로직을 직접 미들웨어로 만들 수 있습니다.  
`@before_agent`, `@after_agent`, `@wrap_model_call` 데코레이터를 사용해 어느 시점에나 끼워 넣을 수 있습니다.

### node-style — 입력 검증 가드레일

`can_jump_to=["end"]`를 사용하면 미들웨어에서 에이전트 실행을 **즉시 종료**할 수 있습니다.  
응답을 `{"messages": [...], "jump_to": "end"}` 형태로 반환하면 LLM을 호출하지 않고 바로 응답합니다.

In [10]:
import re
from langchain.agents.middleware import before_agent
from langchain.messages import AIMessage

# 차단할 시크릿 패턴 (OpenAI/Anthropic, AWS, GitHub 토큰 등)
SECRET_PATTERNS = [
    r'sk-[A-Za-z0-9_\-]{20,}',     # OpenAI / Anthropic 스타일 키
    r'AKIA[0-9A-Z]{16}',           # AWS Access Key ID
    r'ghp_[A-Za-z0-9]{36}',        # GitHub Personal Access Token
]

# can_jump_to=['end']: 이 미들웨어는 'end' 노드로 점프할 수 있습니다
# 즉, LLM을 호출하지 않고 즉시 응답을 반환할 수 있습니다
@before_agent(can_jump_to=['end'])
def block_secret_leak(state, runtime):
    """사용자 입력에 API 키/시크릿이 포함되어 있으면 LLM 호출 전에 차단합니다."""
    human_message = state['messages'][-1]
    text = human_message.content
    for pattern in SECRET_PATTERNS:
        if re.search(pattern, text):
            print('⚠️  시크릿 패턴 감지 — LLM 호출을 차단합니다')
            return {
                'messages': [AIMessage(content=(
                    '요청에 API 키 또는 시크릿으로 보이는 값이 포함되어 있어 '
                    '보안을 위해 처리하지 않았습니다. 해당 키를 즉시 폐기(rotate)하고 '
                    '민감하지 않은 정보로 다시 질문해 주세요.'
                ))],
                'jump_to': 'end'  # LLM 호출 없이 즉시 위 메시지로 응답합니다
            }
    return None  # 정상 입력이면 None 반환 → 에이전트 정상 실행


In [11]:
agent = create_agent(
    model='gpt-5-mini',
    middleware=[block_secret_leak],
)

In [12]:
# 정상 입력 — 미들웨어 통과 후 LLM이 답변합니다
agent.invoke({
    "messages": [{"role": "user", "content": "LangChain 미들웨어가 뭐야?"}]
})

{'messages': [HumanMessage(content='LangChain 미들웨어가 뭐야?', additional_kwargs={}, response_metadata={}, id='7e3979a3-75df-4bd9-9d6e-34b8302fe384'),
  AIMessage(content='간단히 말하면: LangChain 미들웨어는 당신의 애플리케이션과 LangChain이 호출하는 LLM/체인/툴 사이에 끼어들어(중간에 개입하여) 요청이나 응답을 관찰·수정·로깅·보호하는 코드 조각입니다. 웹 프레임워크의 미들웨어처럼 체인 실행 흐름을 가로채어 공통 기능(로깅, 캐시, 인증, 프롬프트 변환, 오류 처리 등)을 재사용 가능하게 만듭니다.\n\n무엇을 할 수 있나\n- 프롬프트 자동 전처리(템플릿 삽입, 민감한 정보 마스킹)\n- 응답 후처리(후처리 규칙 적용, 형식 검증)\n- 로깅 및 추적(모든 호출 기록, 성능 측정, 메트릭 전송)\n- 캐싱(동일 입력 재요청 시 LLM 호출 건수 절감)\n- 레이트리밋/재시도 정책 적용\n- 스트리밍 토큰 가로채기(실시간 UI 전송)\n- 감사·컴플라이언스(민감데이터 제거, 사용자 ID 첨부)\n- 툴 호출 전/후 검사 및 보안\n\nLangChain에서 미들웨어 구현 방식(개념적)\n- Callback/Handler: LangChain의 콜백 시스템을 이용해 LLM 이벤트(on_llm_start, on_llm_new_token, on_llm_end, on_chain_start 등)를 가로챕니다.\n- 래퍼/데코레이터 패턴: LLM 객체나 Chain을 감싸 입력/출력을 변형합니다.\n- 전용 미들웨어 API: 일부 LangChain 구현체들은 미들웨어 인터페이스를 제공해 체인에 등록할 수 있게 합니다(버전/언어에 따라 API 차이 있음).\n\n간단한 예시(개념) — Python (콜백 핸들러 방식)\n- LangChain의 BaseCallbackHandler를 상속해 이벤트를 처리할 수 있음.\n\n예시(개념 코드

In [13]:
# API 키가 포함된 입력 — 즉시 보안 경고 메시지 반환, LLM 미호출
res = agent.invoke({
    "messages": [{"role": "user", "content": "내 OpenAI 키 sk-proj-ABCDEFGHIJKLMNOPQRSTUVWX 로 호출해 줘"}]
})

print(res['messages'][-1].content)


⚠️  시크릿 패턴 감지 — LLM 호출을 차단합니다
요청에 API 키 또는 시크릿으로 보이는 값이 포함되어 있어 보안을 위해 처리하지 않았습니다. 해당 키를 즉시 폐기(rotate)하고 민감하지 않은 정보로 다시 질문해 주세요.


### wrap-style — 메시지 길이 기반 동적 모델 선택

사용자 메시지의 **길이**에 따라 자동으로 모델을 선택합니다.  
짧은 질문 → 저렴한 모델, 긴 질문 → 고성능 모델로 비용을 최적화합니다.

In [19]:
from langchain_openai import ChatOpenAI

@wrap_model_call
def dynamic_model_selector(request, handler):
    # 최근 사용자 메시지를 추출합니다
    last_msg = request.messages[-1].content if request.messages else ''
    msg_len = len(last_msg)  # 메시지 글자 수 측정

    # 메시지 길이에 따라 모델을 선택합니다
    if msg_len < 10:
        model_name = 'gpt-5-nano'   # 짧은 질문: 가장 경제적인 모델
    elif msg_len < 30:
        model_name = 'gpt-5-mini'   # 중간 길이: 균형 모델
    else:
        model_name = 'gpt-5'        # 긴 질문: 고성능 모델

    print(f'💬 메시지 길이: {msg_len}자, 선택된 모델: {model_name}')

    # 선택된 모델로 request를 교체하여 LLM을 호출합니다
    new_model = ChatOpenAI(model_name=model_name)
    new_request = request.override(model=new_model)

    return handler(new_request)


In [20]:
agent = create_agent(
    model='gpt-5-mini',       # 기본값 (미들웨어에서 동적으로 교체됩니다)
    tools=[],
    middleware=[dynamic_model_selector],
)

In [21]:
# 짧은 메시지 (9자) — gpt-5-nano 선택
agent.invoke(
    {"messages": [{"role": "user", "content": "안녕하세요"}]},
)

💬 메시지 길이: 5자, 선택된 모델: gpt-5-nano


{'messages': [HumanMessage(content='안녕하세요', additional_kwargs={}, response_metadata={}, id='c58ba128-4ae7-4686-b3a5-19edc07244ee'),
  AIMessage(content='안녕하세요! 무엇을 도와드릴까요?\n\n도움이 될 수 있는 영역 예시\n- 글쓰기: 문장 다듬기, 아이디어 정리\n- 번역: 한국어↔다른 언어 번역이나 다듬기\n- 정보 정리/요약\n- 코딩 문제 풀이나 간단한 프로그래밍 도움\n- 공부 계획이나 학습 자료 만들기\n- 요리 레시피 아이디어나 팁\n- 생활 팁이나 아이디어 브레인스토밍\n\n특정 주제나 목표가 있으면 알려 주세요. 바로 도와드리겠습니다. 예시 요청도 환영합니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 793, 'prompt_tokens': 8, 'total_tokens': 801, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 640, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DM6C2toZdqJVHBTnjfQzWC34BSALm', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d1423-f3e4-7f81-b677-650d3cc

In [22]:
# 중간 길이 메시지 — gpt-5-mini 선택
agent.invoke(
    {"messages": [{"role": "user", "content": "안녕하세요. 저는 Jay입니다."}]},
)

💬 메시지 길이: 17자, 선택된 모델: gpt-5-mini


{'messages': [HumanMessage(content='안녕하세요. 저는 Jay입니다.', additional_kwargs={}, response_metadata={}, id='82719a3a-aef0-4f1e-9152-c9fddca5a1e7'),
  AIMessage(content='안녕하세요 Jay님, 반갑습니다! 저는 도와드릴 준비가 되어 있어요. 어떤 도움을 원하시나요? (한국어·영어 모두 가능합니다.)', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 109, 'prompt_tokens': 13, 'total_tokens': 122, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 64, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DM6CDVpA2qYWUyxBKVKc7nTGmisja', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d1424-1d10-7860-a753-7238fd05d015-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 13, 'output_tokens': 109, 'total_tokens': 122, 'input_token_details': {'audio

In [23]:
# 긴 메시지 — gpt-5 선택
agent.invoke(
    {"messages": [{"role": "user", "content": "안녕하세요. 저는 Jay입니다. 인공지능에 관해 궁금한게 있어요."}]},
)

💬 메시지 길이: 36자, 선택된 모델: gpt-5


{'messages': [HumanMessage(content='안녕하세요. 저는 Jay입니다. 인공지능에 관해 궁금한게 있어요.', additional_kwargs={}, response_metadata={}, id='0fba1a03-8e93-4f74-83cb-8a09319e31e7'),
  AIMessage(content='안녕하세요, Jay님! 반갑습니다. 인공지능에 대해 어떤 점이 가장 궁금하신가요? 아래 중에서 골라주셔도 되고, 바로 첫 질문을 해주셔도 돼요.\n\n- 개념: AI·머신러닝·딥러닝 차이, 지도/비지도/강화학습, 신경망 원리\n- 생성형 AI: LLM(트랜스포머), 프롬프트 설계, 파인튜닝, RAG\n- 실습: 파이썬으로 시작하기, scikit-learn/PyTorch 예제, 간단한 프로젝트\n- 적용: 텍스트 요약/번역, 추천, 이미지 분류, 자동화 에이전트\n- 평가와 한계: 성능 지표, 환각 문제, 안전/프라이버시\n- 윤리/법·저작권: 편향, 개인정보, 규제 이슈\n- 커리어/학습 로드맵: 공부 순서, 자료, 포트폴리오 아이디어\n\n확인하고 싶어요:\n1) 현재 수준은 어떤가요? (완전 초보 / 기초 있음 / 실무)\n2) 목표는? (개념 이해 / 코딩 실습 / 업무 자동화 / 연구)\n3) 파이썬 사용 가능하신가요? 한국어/영어 자료 중 선호는요?\n\n원하시면 바로 짧은 개요부터 시작해드릴게요.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 843, 'prompt_tokens': 26, 'total_tokens': 869, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 512, 'rejected_prediction_tokens': 0}, 'prompt_tokens_det

## Guardrails (안전 장치)

**Guardrails**는 에이전트의 입력/출력을 검사하여 안전하고 적절한 응답만 허용합니다.  
교육용 AI처럼 **제약이 필요한 환경**에서 특히 중요합니다.

| Guardrail 위치 | 동작 |
|--------------|------|
| `@before_agent` | 사용자 입력을 LLM에 보내기 **전에** 검사 |
| `@after_agent` | LLM의 응답을 사용자에게 보내기 **전에** 검사 |

#### Before agent Guardrails — 입력 필터

사용자의 질문이 들어왔을 때 LLM을 호출하기 **전에** 검사합니다.  
부정행위, 학습 방해, 유해 콘텐츠를 카테고리별로 감지하여 교육적 피드백을 제공합니다.

In [24]:
# 카테고리별 차단 키워드 사전
forbidden_topics = {
    'cheating': ['답지', '정답 알려줘', '숙제 대신', '써줘', '베끼기'],  # 부정행위 관련
    'distraction': ['롤', '게임', '유튜브', '아이돌', '웹툰'],            # 학습 방해 요소
    'harmful': ['담배', '술', '폭력', '싸움', '바보']                     # 유해 콘텐츠
}

In [25]:
from langchain.agents.middleware import before_agent

@before_agent(can_jump_to=['end'])
def education_guardrail(state, runtime):
    """
    학생의 질문 의도를 파악하여 교육적이지 않거나 부정행위가 의심될 경우,
    LLM에게 질문을 넘기지 않고 교육적인 멘트로 즉시 교정합니다.
    """
    # 1. 메시지 유효성 검사
    if not state['messages']:
        return None

    last_message = state['messages'][-1]
    if last_message.type != 'human':
        return None

    user_text = last_message.content

    # 2. 카테고리별 키워드 검사 (카테고리마다 다른 피드백 제공)
    for keyword in forbidden_topics['cheating']:
        if keyword in user_text:
            return {
                'messages': [AIMessage(content=f"'{keyword}'와 관련된 요청은 도와드리기 어렵습니다. 스스로 해결해보세요!")],
                'jump_to': 'end'
            }

    for keyword in forbidden_topics['distraction']:
        if keyword in user_text:
            return {
                'messages': [AIMessage(content=f"지금은 학습 시간입니다. '{keyword}' 대신 공부에 집중해볼까요?")],
                'jump_to': 'end'
            }

    for keyword in forbidden_topics['harmful']:
        if keyword in user_text:
            return {
                'messages': [AIMessage(content='적절하지 않은 표현이 포함되어 있습니다. 정중하게 질문해 주세요.')],
                'jump_to': 'end'
            }

    return None  # 문제 없는 질문이면 LLM에게 전달합니다


In [26]:
from langchain.agents import create_agent

agent = create_agent(
    model='gpt-5-nano',
    tools=[],
    middleware=[education_guardrail],
)

In [27]:
# 정상 학습 질문 — 가드레일 통과, LLM이 답변합니다
agent.invoke({
    "messages": [{"role": "user", "content": "피타고라스의 정리가 이해가 안 돼. 설명해줘."}]
})

{'messages': [HumanMessage(content='피타고라스의 정리가 이해가 안 돼. 설명해줘.', additional_kwargs={}, response_metadata={}, id='47026c84-7c8c-48ec-aa98-06ecafd3a7d8'),
  AIMessage(content='좋아요! 피타고라스의 정리는 직각삼각형에 관한 아주 간단한 규칙이에요.\n\n1) 기본 정리\n- 직각삼각형에서 두 직각 변(밑변과 높이)을 각각 a와 b, 빗변(가장 긴 변)을 c라고 할게요.\n- 그러면 a^2 + b^2 = c^2가 성립합니다.\n- 예를 들어, 3-4-5 삼각형에서는 3^2 + 4^2 = 9 + 16 = 25 = 5^2이 되죠.\n\n2) 왜 그런지 intuitively 이해하는 방법\n- 직각 삼각형의 각 변에 "제곱의 면적"을 생각해보면 이해가 쉬워요.\n- 빗변의 제곱은 나머지 두 변의 제곱의 합과 같다 = 빗변으로 만들어진 큰 제곱의 면적이, 두 작은 직각삼각형의 면적 합과 같다는 뜻이에요.\n\n3) 두 가지 쉬운 증명 아이디어\n- 아이디어 A: 면적을 이용한 증명\n  - 한 변의 길이가 (a + b)인 큰 정사각형 안에 네 개의 같은 직각삼각형을 대치해 넣으면, 가운데 남는 모양은 빗변 길이 c인 정사각형 모양이 됩니다.\n  - 큰 정사각형의 면적은 (a + b)^2이고, 네 개의 삼각형이 차지하는 면적은 4 × (1/2)ab = 2ab이에요.\n  - 그러면 남은 중앙 부분의 면적은 (a + b)^2 − 2ab = a^2 + b^2가 되고, 이 중앙 부분의 면적은 빗변의 제곱 c^2과 같으므로 a^2 + b^2 = c^2이 됩니다.\n- 아이디어 B: 닮음(비례)을 이용한 증명\n  - 직각삼각형에서 빗변에 직선을 내려 높이를 만들면, 생긴 두 작은 삼각형도 모두 원래 삼각형과 닮습니다.\n  - 이 닮음 관계를 이용하면 a^2 = c·p, b^2 = c·q 이 되고, p + q = c이므로 a^2 + b^2 = c

In [28]:
# 부정행위 감지 — LLM 미호출, 즉시 교육적 피드백 반환
agent.invoke({
    "messages": [{"role": "user", "content": "나 독후감 쓰기 귀찮은데 숙제 대신 써줘."}]
})

{'messages': [HumanMessage(content='나 독후감 쓰기 귀찮은데 숙제 대신 써줘.', additional_kwargs={}, response_metadata={}, id='7f4124ad-6bb0-4b70-9e3b-8ba93ef37373'),
  AIMessage(content="'숙제 대신'와 관련된 요청은 도와드리기 어렵습니다. 스스로 해결해보세요!", additional_kwargs={}, response_metadata={}, id='7c92c35e-05c3-41c5-8443-7366e52ca39e', tool_calls=[], invalid_tool_calls=[])]}

In [29]:
# 학습 방해 감지
agent.invoke({
    "messages": [{"role": "user", "content": "아 공부하기 싫다. 롤 관련 유튜브 영상이나 찾아줘"}]
})

{'messages': [HumanMessage(content='아 공부하기 싫다. 롤 관련 유튜브 영상이나 찾아줘', additional_kwargs={}, response_metadata={}, id='3663bd3b-1409-41eb-9e07-6157a63ee041'),
  AIMessage(content="지금은 학습 시간입니다. '롤' 대신 공부에 집중해볼까요?", additional_kwargs={}, response_metadata={}, id='d0016168-045d-4d6a-b7f3-8c204f8771ee', tool_calls=[], invalid_tool_calls=[])]}

In [30]:
# 유해 표현 감지
agent.invoke({
    "messages": [{"role": "user", "content": "아 너 왜이렇게 바보같냐..."}]
})

{'messages': [HumanMessage(content='아 너 왜이렇게 바보같냐...', additional_kwargs={}, response_metadata={}, id='19674cce-eab0-43f5-9514-46835c199130'),
  AIMessage(content='적절하지 않은 표현이 포함되어 있습니다. 정중하게 질문해 주세요.', additional_kwargs={}, response_metadata={}, id='4ce385db-7b44-4baf-bcff-f138cdb834d5', tool_calls=[], invalid_tool_calls=[])]}

#### After agent Guardrails — 출력 필터

LLM이 응답을 생성한 **직후** 사용자에게 보내기 전에 검사합니다.  
**감시자 LLM(auditor)**이 메인 LLM의 응답을 평가하여 부적절한 응답을 교정합니다.

> 💡 LLM으로 LLM을 감시하는 패턴입니다. 규칙 기반 필터보다 훨씬 유연합니다.

In [31]:
from langchain.chat_models import init_chat_model

# 감시자 역할을 할 별도의 LLM 모델을 준비합니다
safety_model = init_chat_model('gpt-5-nano')

In [32]:
from langchain.agents.middleware import after_agent


@after_agent
def answer_leakage_guardrail(state, runtime):
    """
    AI가 답변을 생성한 '직후', 사용자에게 보여주기 전에 내용을 검사합니다.
    만약 AI가 문제의 정답을 직접 말해버렸다면 이를 감지하고 수정합니다.
    """
    # 1. 메시지 유효성 검사
    if not state['messages']:
        return None
    last_message = state['messages'][-1]

    # 마지막 메시지가 AI의 답변이 아니면 검사 불필요
    if not isinstance(last_message, AIMessage):
        return None

    # 2. 감시자 LLM에게 응답의 교육적 적절성을 평가하도록 요청합니다
    auditor_prompt = f"""
    당신은 엄격한 교육 감독관입니다.
    다음 '튜터의 답변'을 확인하세요.

    튜터의 답변:
    {last_message.content}

    판단 기준:
    - 수학 문제의 '최종 숫자 정답'을 직접 알려준 경우: 'FAIL'
    - 개념 설명, 힌트 제공, 풀이 과정 안내는: 'PASS'

    'PASS' 또는 'FAIL' 중 하나만 답하세요.
    """

    audit_result = safety_model.invoke(auditor_prompt).content.strip()
    print(f'감사 결과: {audit_result}')

    # 3. FAIL이면 교육적인 힌트 메시지로 교체합니다
    if 'FAIL' in audit_result:
        return {
            'messages': [
                *state['messages'][:-1],  # 기존 메시지 유지
                AIMessage(content='정답을 직접 알려드리기 어렵습니다. 피타고라스 정리(a²+b²=c²)를 사용해 직접 계산해보세요!')
            ]
        }

    return None  # PASS이면 원래 응답 그대로 반환


In [33]:
agent = create_agent(
    model='gpt-5',              # 고성능 모델 사용 (설명 능력 중요)
    tools=[],
    middleware=[answer_leakage_guardrail],
)

In [34]:
# 정답 직접 요구 — after_agent 가드레일이 응답을 검사하여 힌트로 교체합니다
agent.invoke({
    "messages": [{"role": "user", "content": "직각 삼각형 두 직각변의 길이가 3과 4라면 빗변의 길이가 뭐야? 이 문제 너무 어려워. 그냥 정답 알려줘."}]
})

감사 결과: FAIL


{'messages': [HumanMessage(content='직각 삼각형 두 직각변의 길이가 3과 4라면 빗변의 길이가 뭐야? 이 문제 너무 어려워. 그냥 정답 알려줘.', additional_kwargs={}, response_metadata={}, id='1f404b6b-f261-42ad-8b29-8e832aea9822'),
  AIMessage(content='5', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 138, 'prompt_tokens': 46, 'total_tokens': 184, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DM6DBEV9Yx5esriliMrdXj2Ue6Sef', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d1425-0619-7e33-97a1-e660f01366e3-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 46, 'output_tokens': 138, 'total_tokens': 184, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output

#### Combine multiple Guardrails — 다중 가드레일 조합

여러 미들웨어를 순서대로 쌓아 **다층 방어 구조**를 만들 수 있습니다.

```
입력 → [Layer 1: 규칙 필터] → [Layer 2: 개인정보 보호] → [Layer 3: 상담 이관] → LLM → [Layer 4: 출력 검사] → 응답
```

In [35]:
import re

# Layer 2: 개인정보(전화번호, 이메일) 자동 마스킹
@before_agent
def student_safety_middleware(state, runtime):
    """학생의 전화번호나 이메일이 감지되면 마스킹 처리하여 안전을 확보합니다."""
    if not state['messages']:
        return None
    last_message = state['messages'][-1]
    if last_message.type != 'human':
        return None

    content = last_message.content

    # 전화번호 패턴 감지 및 마스킹
    phone_pattern = r'01[016789]-?[0-9]{3,4}-?[0-9]{4}'
    email_pattern = r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}'

    is_redacted = False

    if re.search(phone_pattern, content):
        content = re.sub(phone_pattern, '[전화번호 마스킹됨]', content)
        is_redacted = True

    if re.search(email_pattern, content):
        content = re.sub(email_pattern, '[이메일 마스킹됨]', content)
        is_redacted = True

    if is_redacted:
        print('⚠️ 개인정보가 감지되어 마스킹 처리되었습니다.')
        # 마스킹된 내용으로 메시지를 교체합니다
        from langchain.messages import HumanMessage
        new_messages = state['messages'][:-1] + [HumanMessage(content=content)]
        return {'messages': new_messages}

    return None


In [36]:
# 심리적 위기 상황 키워드 목록
ESCALATION_KEYWORDS = ['왕따', '괴롭힘', '우울해', '학교 폭력', '상담 선생님', '사람 불러줘']

# Layer 3: 심리적 위기 감지 시 AI 중단 + 인간 상담사 알림
@before_agent(can_jump_to=['end'])
def counseling_escalation_middleware(state, runtime):
    """심리적 위기 상황이나 상담 요청이 감지되면 AI 답변을 멈추고 인간 상담사에게 알림을 보냅니다."""
    if not state['messages']:
        return None
    last_message = state['messages'][-1]

    for keyword in ESCALATION_KEYWORDS:
        if keyword in last_message.content:
            print(f'✋ [상담 이관] 심각한 고민/요청 감지: {keyword}')
            # 실제 서비스에서는 여기서 상담 교사에게 Slack/Email 알림을 보냅니다
            return {
                'messages': [AIMessage(content='혼자 힘든 마음을 나눠줘서 고마워요. 선생님께서 도와주실 거예요. 잠시만 기다려주세요.')],
                'jump_to': 'end'
            }

    return None


In [37]:
# 4중 방어막 에이전트 (미들웨어는 리스트 순서대로 실행됩니다)
agent = create_agent(
    model='gpt-5-nano',
    tools=[],
    middleware=[
        education_guardrail,              # Layer 1: 입력 필터 (부정행위/학습방해)
        student_safety_middleware,        # Layer 2: 개인정보 보호 (전화번호/이메일)
        counseling_escalation_middleware, # Layer 3: 상담 이관 (위기 대응)
        answer_leakage_guardrail          # Layer 4: 출력 필터 (정답 유출 방지)
    ],
)

In [38]:
# 개인정보 포함 — Layer 2가 전화번호를 마스킹합니다
agent.invoke({
    "messages": [{"role": "user", "content": "저 수학 과외 구하고 싶어요. 제 번호 010-1234-5678로 연락 주세요."}]
})

⚠️ 개인정보가 감지되어 마스킹 처리되었습니다.
감사 결과: PASS


{'messages': [HumanMessage(content='저 수학 과외 구하고 싶어요. 제 번호 010-1234-5678로 연락 주세요.', additional_kwargs={}, response_metadata={}, id='662de2fb-de0c-45ae-9f54-453a09797042'),
  HumanMessage(content='저 수학 과외 구하고 싶어요. 제 번호 [전화번호 마스킹됨]로 연락 주세요.', additional_kwargs={}, response_metadata={}, id='e05a6171-81a1-4c21-aaef-12f99a29f67d'),
  AIMessage(content='도와드릴 수 있어요. 다만 제가 직접 전화나 메시지로 연락해 드리지는 못합니다. 대신 수학 과외를 구할 때 쓸 수 있는 문구와 방법을 준비해드릴게요. 아래 템플릿 중 상황에 맞게 골라서 사용하시고, 필요하면 더 맞춤화해 드리겠습니다.\n\n1) 온라인/오프라인 구인 글 예시 (게시글용)\n제목: 수학 과외 선생님 구합니다(중/고등 수학)\n\n본문 예시:\n- 현재 학년/수준: 예) 중등 2학년 수학 또는 고등 수학 기초 다지기\n- 목표: 예) 기초 개념 보강, 수능/학력평가 대비, 기출 문제 풀이 강화\n- 수업 방식: 오프라인 또는 온라인 가능 여부\n- 주당 수업 시간: 예) 주 2회, 각 90분\n- 지역/가능 지역: 예) 서울 강남구, 경기 남양주 등\n- 예산: 예) 1회당 XX 원 내외(협의 가능)\n- 자격 요건: 경력 우대, 교사 자격증/대학원생 우대, 수학 과목에 대한 명확한 설명 가능 등\n- 연락 방법: 연락처 [전화번호] 또는 메시지로 간단한 소개 부탁드립니다\n- 기타: 진도표 제공 여부, 진단 평가 가능 여부, 트라이얼 수업 여부 등\n\n2) 문자/메시지용 간단한 템플릿\n안녕하세요. 수학 과외를 구합니다.\n- 학년/수준: [예: 중등 2학년]\n- 목표: [예: 기초 개념 보강, 기출 풀이]\n- 원하시는 방식: 

In [39]:
# 심리적 위기 표현 — Layer 3이 감지하여 즉시 상담사 이관 메시지 반환
agent.invoke({
    "messages": [{"role": "user", "content": "나 요즘 학교에서 왕따 당하는 것 같아서 너무 우울해."}]
})

✋ [상담 이관] 심각한 고민/요청 감지: 왕따
감사 결과: PASS


{'messages': [HumanMessage(content='나 요즘 학교에서 왕따 당하는 것 같아서 너무 우울해.', additional_kwargs={}, response_metadata={}, id='9f0c3de3-94b3-47a9-bfa2-303fa00c9ef3'),
  AIMessage(content='혼자 힘든 마음을 나눠줘서 고마워요. 선생님께서 도와주실 거예요. 잠시만 기다려주세요.', additional_kwargs={}, response_metadata={}, id='dd6b1955-3ac4-48d6-97aa-24765cbc5a40', tool_calls=[], invalid_tool_calls=[])]}

## Long-term Memory (장기 메모리)

**Checkpointer**는 세션 내 대화를 저장하는 **단기 메모리**입니다.  
**Store**는 여러 세션에 걸쳐 영구적으로 유지되는 **장기 메모리**입니다.

| | Checkpointer | Store |
|---|---|---|
| 저장 범위 | 하나의 thread_id 내 | 여러 thread_id에 걸쳐 유지 |
| 수명 | 세션 단위 | 영구 (InMemoryStore는 프로그램 종료까지) |
| 용도 | 대화 히스토리 | 사용자 프로필, 선호도, 누적 정보 |

### 장기 메모리 생성

In [40]:
# InMemoryStore: 딕셔너리 기반 메모리 저장소
# 실제 서비스에서는 Redis, PostgreSQL 등 영구 DB로 교체합니다
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()


In [41]:
# namespace: 메모리를 구분하는 계층적 키 (사용자ID + 앱이름)
# 같은 사용자라도 앱이 다르면 다른 네임스페이스로 분리합니다
user_id = 'user_001'
application_context = 'personal_assistant'

namespace = (user_id, application_context)  # 튜플로 계층 구성


In [42]:
# store.put(namespace, key, value): 메모리를 저장합니다
# key는 메모리 항목의 고유 식별자입니다
store.put(
    namespace,
    'memory_001',           # 고유 키
    {
        'facts': [
            '사용자는 커피보다 차를 선호함',
            '사용자는 매일 아침 6시에 일어남',
        ],
        'language': 'Korean',
    },
)


In [43]:
# store.get(namespace, key): 특정 키의 메모리를 조회합니다
item = store.get(namespace, 'memory_001')
print('저장된 메모리:', item.value)

저장된 메모리: {'facts': ['사용자는 커피보다 차를 선호함', '사용자는 매일 아침 6시에 일어남'], 'language': 'Korean'}


In [44]:
# store.search(namespace): 해당 네임스페이스의 모든 메모리를 검색합니다
items = store.search(namespace)

In [45]:
items

[Item(namespace=['user_001', 'personal_assistant'], key='memory_001', value={'facts': ['사용자는 커피보다 차를 선호함', '사용자는 매일 아침 6시에 일어남'], 'language': 'Korean'}, created_at='2026-03-22T06:05:29.594700+00:00', updated_at='2026-03-22T06:05:29.594700+00:00', score=None)]

In [46]:
# 두 번째 메모리 항목을 추가합니다
store.put(
    namespace,
    'memory_002',
    {
        'facts': [
            '사용자는 그림 회화 작품을 좋아함',
            '빈센트 반 고흐의 작품을 특히 좋아함',
        ]
    },
)

In [47]:
item = store.get(namespace, 'memory_002')
print('저장된 메모리:', item.value)

저장된 메모리: {'facts': ['사용자는 그림 회화 작품을 좋아함', '빈센트 반 고흐의 작품을 특히 좋아함']}


In [48]:
# 네임스페이스 전체 검색 — 이제 2개의 메모리가 있습니다
items = store.search(namespace)

In [49]:
items

[Item(namespace=['user_001', 'personal_assistant'], key='memory_001', value={'facts': ['사용자는 커피보다 차를 선호함', '사용자는 매일 아침 6시에 일어남'], 'language': 'Korean'}, created_at='2026-03-22T06:05:29.594700+00:00', updated_at='2026-03-22T06:05:29.594700+00:00', score=None),
 Item(namespace=['user_001', 'personal_assistant'], key='memory_002', value={'facts': ['사용자는 그림 회화 작품을 좋아함', '빈센트 반 고흐의 작품을 특히 좋아함']}, created_at='2026-03-22T06:05:33.090007+00:00', updated_at='2026-03-22T06:05:33.090007+00:00', score=None)]

In [50]:
from dataclasses import dataclass

# 에이전트 실행 컨텍스트 — 누가(user_id), 어떤 앱(app_name)으로 실행하는지 식별
@dataclass
class Context:
    user_id: str   # 사용자 고유 ID — store의 namespace에 사용됩니다
    app_name: str  # 앱 이름 — 같은 사용자의 앱별 메모리를 분리합니다

In [51]:
from langchain.agents.middleware import wrap_model_call
from langchain.messages import HumanMessage, SystemMessage

@wrap_model_call
def inject_memory(request, handler):
    # 현재 실행 컨텍스트에서 사용자 정보를 가져옵니다
    current_user = request.runtime.context.user_id
    current_app = request.runtime.context.app_name

    # Store에서 이 사용자의 장기 메모리를 모두 검색합니다
    memories = request.runtime.store.search((current_user, current_app))

    memory_content = '기록된 정보 없음'
    if memories:
        # 검색된 메모리에서 facts 목록을 추출합니다
        extracted_facts = []
        for item in memories:
            if 'facts' in item.value:
                extracted_facts.extend(item.value['facts'])
        memory_content = '\n'.join([f'- {f}' for f in extracted_facts])

    # 장기 메모리를 시스템 프롬프트로 주입합니다
    system_prompt = f'다음은 사용자에 대해 알고 있는 정보입니다:\n{memory_content}'
    new_request = request.override(system_prompt=system_prompt)

    return handler(new_request)


In [52]:
from langchain.agents import create_agent

agent = create_agent(
    model='gpt-5-nano',
    store=store,            # 장기 메모리 저장소 연결
    context_schema=Context,
    middleware=[inject_memory]  # 메모리 주입 미들웨어
)

In [53]:
# user_001의 메모리(차 선호, 아침 6시 기상 등)가 주입되어 에이전트가 알고 있습니다
response = agent.invoke(
    {"messages": [{"role": "user", "content": "나에 대해 알고있는 정보 알려줘"}]},
    context=Context(user_id='user_001', app_name='personal_assistant')
)

In [54]:
# 응답 확인
response

{'messages': [HumanMessage(content='나에 대해 알고있는 정보 알려줘', additional_kwargs={}, response_metadata={}, id='9ef6c029-e076-4bfc-9aff-4b725677ed6e'),
  AIMessage(content='다음 정보를 알고 있어요:\n- 커피보다 차를 선호합니다.\n- 매일 아침 6시에 일어납니다.\n- 그림 회화 작품을 좋아합니다.\n- 특히 빈센트 반 고흐의 작품을 좋아합니다.\n\n필요하시면 이 정보를 바탕으로 차 추천이나 반 고흐 테마의 미술 콘텐츠, 아침 루틴 아이디어 등 맞춤 제안도 드릴게요. 더 알려주실 정보가 있나요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 816, 'prompt_tokens': 80, 'total_tokens': 896, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 704, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DM6EP315P4TIF2L8WYc1p6iJlzsft', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d1426-2dac-7d31-bd4a-e44721216c40-0', tool_calls=[], invalid_tool_call

### Tool 기반 장기 메모리 관리

미들웨어로 메모리를 자동 주입하는 방식 대신, **에이전트가 직접 Tool을 사용**하여 메모리를 읽고 쓸 수 있습니다.  
LLM이 상황에 따라 자율적으로 저장/조회 여부를 판단합니다.

In [55]:
from dataclasses import dataclass

@dataclass
class Context:
    user_id: str
    app_name: str

# Store를 새로 초기화합니다 (이전 예시와 독립적)
store = InMemoryStore()

In [56]:
# from typing import TypedDict
from typing_extensions import TypedDict  # Python 3.12 이하에서 사용

# LLM이 저장해야 할 정보의 구조를 TypedDict로 정의합니다
class UserInfo(TypedDict):
    personal_information: str  # 개인 정보 (이름, 나이, 직업 등)
    preference: str             # 선호도 (좋아하는 것, 싫어하는 것 등)


In [57]:
from langchain_core.runnables import RunnableConfig
from langchain.tools import tool, ToolRuntime

# 사용자 정보를 조회하는 Tool
@tool
def get_user_info(runtime) -> str:
    """
    현재 사용자의 저장된 정보를 조회합니다 (시스템 내부용 도구)
    """
    user_id = runtime.context.user_id
    app = runtime.context.app_name

    # 해당 사용자의 네임스페이스에서 모든 메모리를 검색합니다
    memories = runtime.store.search((user_id, app))

    if not memories:
        return '기록된 정보 없음'

    results = []
    for item in memories:
        data = item.value
        if 'personal_information' in data:
            results.append(f'개인정보: {data["personal_information"]}')
        if 'preference' in data:
            results.append(f'선호도: {data["preference"]}')

    return '\n'.join(results) if results else '기록된 정보 없음'


In [58]:
import uuid

# 사용자 정보를 저장하는 Tool
@tool
def save_user_info(user_info: UserInfo, runtime) -> str:
    """
    사용자의 정보를 저장하거나 업데이트합니다
    """
    user_id = runtime.context.user_id
    app = runtime.context.app_name
    store = runtime.store

    # 고유한 키(UUID)로 새로운 메모리 항목을 저장합니다
    memory_key = str(uuid.uuid4())
    store.put((user_id, app), memory_key, user_info)

    return f'정보가 안전하게 저장되었습니다. (ID: {memory_key})'

In [59]:
from langchain.agents import create_agent

# 메모리 Tool들을 에이전트에 등록합니다
# LLM이 대화 내용을 판단하여 자율적으로 save_user_info/get_user_info를 호출합니다
agent = create_agent(
    model='gpt-5',
    tools=[get_user_info, save_user_info],
    store=store,  # 에이전트에 Store를 연결합니다
    context_schema=Context
)

In [60]:
# 사용자가 이름과 선호도를 알려줍니다 — 에이전트가 자동으로 save_user_info를 호출합니다
response = agent.invoke(
    {"messages": [{"role": "user", "content": "내 이름은 이제 'Alice'야. 커피보단 차를 좋아해"}]},
    context=Context(user_id='user_001', app_name='personal_assistant')
)

In [61]:
response

{'messages': [HumanMessage(content="내 이름은 이제 'Alice'야. 커피보단 차를 좋아해", additional_kwargs={}, response_metadata={}, id='608d3730-ed30-49e2-a36b-d064075e9c2d'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 369, 'prompt_tokens': 188, 'total_tokens': 557, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 320, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DM6EbQR9LaGSYMBmeiMum67M7N9xO', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d1426-5ff0-7d42-8f49-edd899a912ab-0', tool_calls=[{'name': 'save_user_info', 'args': {'user_info': {'personal_information': '이름: Alice', 'preference': '커피보다 차를 더 선호함'}, 'runtime': {}}, 'id': 'call_7y3HbpQUh4ZAMAbGnWlhi4z1', 'type': 'tool_ca

In [62]:
# 저장된 정보를 조회합니다 — 에이전트가 get_user_info를 호출하여 응답합니다
response = agent.invoke(
    {"messages": [{"role": "user", "content": "나에 대해 아는 정보 말해줘"}]},
    context=Context(user_id='user_001', app_name='personal_assistant')
)

In [63]:
response

{'messages': [HumanMessage(content='나에 대해 아는 정보 말해줘', additional_kwargs={}, response_metadata={}, id='fc725d2f-1cc4-4946-ba6c-84af9250ad90'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 216, 'prompt_tokens': 180, 'total_tokens': 396, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 192, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DM6EwTPk2LKF7OAP44fkDLKQ77P0X', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d1426-b268-7883-b4fd-18991fd42cd6-0', tool_calls=[{'name': 'get_user_info', 'args': {'runtime': 'now'}, 'id': 'call_YgpqXEzKMahXwLbSynuCdgtw', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 180, 'output_tokens': 216, 'total_

In [64]:
# 다른 사용자(user_002)로 조회 — user_001의 정보가 없습니다 (사용자별 격리 확인)
response = agent.invoke(
    {"messages": [{"role": "user", "content": "나에 대해 아는 정보 말해줘"}]},
    context=Context(user_id='user_002', app_name='personal_assistant')
)

In [65]:
response

{'messages': [HumanMessage(content='나에 대해 아는 정보 말해줘', additional_kwargs={}, response_metadata={}, id='7e455491-6d89-4108-bb7d-6d8acb85314c'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 152, 'prompt_tokens': 180, 'total_tokens': 332, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DM6FCFCUttifQYklEfv3nfgHogjVX', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d1426-f067-7701-b977-f065246f5938-0', tool_calls=[{'name': 'get_user_info', 'args': {'runtime': 'current'}, 'id': 'call_2kH7qdW45a1vALW3QKJG9KqV', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 180, 'output_tokens': 152, 'to